# CPML Absorption Verification

This notebook checks that CPML is transparent before a boundary return can
arrive, substantially suppresses late reflected energy, remains effective
across several thicknesses, and is excluded from material gradients.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


Repository root: /Users/llsra/Desktop/DeepGPR
DeepGPR package: /Users/llsra/Desktop/DeepGPR/src/DeepGPR/__init__.py


In [2]:
import torch

DEVICE = torch.device("cpu")
CHECKS = []
METADATA = vu.runtime_metadata(DeepGPR, DEVICE)
nx, ny, nt = 64, 80, 1200
dx, dt = 0.02, 2.0e-11
er_base = torch.full((nx, ny), 4.0)
se_base = torch.zeros_like(er_base)
source_location = torch.tensor([[[32, 40, 0]]], dtype=torch.int32)
receiver_location = torch.tensor([[[32, 42, 0]]], dtype=torch.int32)
source = DeepGPR.wavelet.ricker(4.0e8, nt, dt, 2.5e-9).reshape(1, nt, 1)

def simulate(pml, er=er_base, se=se_base):
    return DeepGPR.compute(
        device=DEVICE,
        dx=dx,
        dt=dt,
        source_amplitudes=source,
        source_location=source_location,
        receiver_location=receiver_location,
        er=er,
        se=se,
        pmlthick=pml,
        fdtd_order=2,
        mode=2,
    )

no_pml = simulate(0)[-1]
pml_10 = simulate(10)[-1]
early_stop = 300
early_error = vu.relative_l2(
    pml_10[:, :early_stop], no_pml[:, :early_stop]
)
vu.record_check(
    CHECKS,
    "CPML does not alter the causal early interior trace",
    early_error < 2.0e-6,
    relative_l2=early_error,
    early_stop_sample=early_stop,
    tolerance=2.0e-6,
)


[PASS] CPML does not alter the causal early interior trace
{
  "early_stop_sample": 300,
  "relative_l2": 1.9321518928760893e-09,
  "tolerance": 2e-06
}


In [3]:
late_start = 600
reference_late_rms = vu.signal_rms(no_pml[:, late_start:])
thickness_rows = []
for thickness in (6, 10, 14):
    response = pml_10 if thickness == 10 else simulate(thickness)[-1]
    late_rms = vu.signal_rms(response[:, late_start:])
    ratio = late_rms / max(reference_late_rms, 1.0e-30)
    row = {
        "pml_thickness": thickness,
        "late_start_sample": late_start,
        "late_rms": late_rms,
        "no_pml_late_rms": reference_late_rms,
        "reflection_rms_ratio": ratio,
    }
    thickness_rows.append(row)
    vu.record_check(
        CHECKS,
        f"late reflected energy is suppressed for CPML thickness {thickness}",
        ratio < 0.25,
        **row,
        tolerance=0.25,
    )


[PASS] late reflected energy is suppressed for CPML thickness 6
{
  "late_rms": 0.02747862763760151,
  "late_start_sample": 600,
  "no_pml_late_rms": 156.9486719825706,
  "pml_thickness": 6,
  "reflection_rms_ratio": 0.00017508034499745913,
  "tolerance": 0.25
}
[PASS] late reflected energy is suppressed for CPML thickness 10
{
  "late_rms": 0.027889762739531344,
  "late_start_sample": 600,
  "no_pml_late_rms": 156.9486719825706,
  "pml_thickness": 10,
  "reflection_rms_ratio": 0.0001776998963242489,
  "tolerance": 0.25
}
[PASS] late reflected energy is suppressed for CPML thickness 14
{
  "late_rms": 0.027804432962260137,
  "late_start_sample": 600,
  "no_pml_late_rms": 156.9486719825706,
  "pml_thickness": 14,
  "reflection_rms_ratio": 0.00017715621681302192,
  "tolerance": 0.25
}


In [4]:
er = er_base.clone().requires_grad_(True)
se = torch.full_like(er, 2.0e-4, requires_grad=True)
gradient_result = simulate(10, er=er, se=se)
gradient_result[-1].square().mean().backward()
vu.assert_finite("CPML model gradients", er.grad, se.grad)
boundary = vu.pml_boundary_mask(er.shape, 10, DEVICE)
interior = ~boundary
er_boundary_absmax = vu.boundary_absmax(er.grad, boundary)
se_boundary_absmax = vu.boundary_absmax(se.grad, boundary)
er_interior_absmax = float(er.grad[interior].abs().max())
se_interior_absmax = float(se.grad[interior].abs().max())
vu.record_check(
    CHECKS,
    "relative-permittivity gradient is exactly zero in CPML cells",
    er_boundary_absmax == 0.0 and er_interior_absmax > 0.0,
    boundary_absmax=er_boundary_absmax,
    interior_absmax=er_interior_absmax,
)
vu.record_check(
    CHECKS,
    "conductivity gradient is exactly zero in CPML cells",
    se_boundary_absmax == 0.0 and se_interior_absmax > 0.0,
    boundary_absmax=se_boundary_absmax,
    interior_absmax=se_interior_absmax,
)


[PASS] relative-permittivity gradient is exactly zero in CPML cells
{
  "boundary_absmax": 0.0,
  "interior_absmax": 655.3296508789062
}
[PASS] conductivity gradient is exactly zero in CPML cells
{
  "boundary_absmax": 0.0,
  "interior_absmax": 14167.8984375
}


In [5]:
vu.save_report(
    "02_cpml_absorption",
    CHECKS,
    METADATA,
    extra={"thickness_rows": thickness_rows},
)
print(f"Completed {len(CHECKS)} required checks.")


Report written to /Users/llsra/Desktop/DeepGPR/tests/results/02_cpml_absorption.json
Completed 6 required checks.
